## Data Augmentation with Galaxy10 DECals Dataset

The Galaxy10 DECals dataset consists of 17,736 256 $\times$ 256 images of galaxies classified into 10 categories. The underlying images were originally sourced from the _Sloan Digital Sky Survey (SDSS)_ via the Galaxy Zoo project. Here we use the dataset to train a galaxy type classifier and illustrate the impact of data augmentation.

In [ ]:
import torch
import torchvision
import torchvision.transforms.functional as TF
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import transforms
import numpy as np
from tqdm.notebook import trange
import h5py
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

### Load and Visualize the Data

In [ ]:
with h5py.File('Galaxy10_DECals.h5', 'r') as gF:
    images = np.array(gF['images'])
    labels = np.array(gF['ans'])

In [ ]:
from sklearn.utils import shuffle
images, labels = shuffle(images, labels)

In [ ]:
gal10_classes = ['Disturbed', 'Merging', 'Round Smooth',\
                 'In-Bteween Round Smooth', 'Cigar-shaped Smooth', \
                 'Barred Spiral', 'Unbarred Tight Spiral', \
                 'Unbarred Loose Spiral', 'Edge-on without Bulge', \
                 'Edge-on with Bulge']
fig, ax = plt.subplots(2, 5, figsize=(10,5))


for i in range(2):
    for j in range(5):
        img = images[i * 5 + j]
        ax[i, j].axes.xaxis.set_visible(False)
        ax[i, j].axes.yaxis.set_visible(False)
        ax[i, j].imshow(img)
        cl = labels[i * 5 + j]
        ax[i, j].set_title(gal10_classes[cl])

fig.savefig('Galaxy10DECals.png', dpi=300, bbox_inches='tight')

### Prepare the Data

The data must be prepared in a similar fashion to the MNIST examples except now there are three channels instead of one. The classes must be converted to a categorical variable.

In [ ]:
train_idx, test_idx = train_test_split(np.arange(labels.shape[0]), test_size=0.1)
X_train, y_train, X_test, y_test = images[train_idx], labels[train_idx], images[test_idx], labels[test_idx]

In [ ]:
encoder = OneHotEncoder(sparse_output=False)
train_y = encoder.fit_transform(y_train.reshape(-1, 1))
test_y = encoder.transform(y_test.reshape(-1, 1))

In [ ]:
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')
X_train /= 255
X_test /= 255

In [ ]:
X_train = torch.Tensor(X_train).permute(0, 3, 1, 2).to(device)
train_y = torch.Tensor(train_y).to(device)
train_dataset = TensorDataset(X_train, train_y)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, 
                                           shuffle=True, drop_last=True)

X_test = torch.Tensor(X_test).permute(0, 3, 1, 2).to(device)
test_y = torch.Tensor(test_y).to(device)
test_dataset = TensorDataset(X_test, test_y)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, 
                                          shuffle=False, drop_last=True)

### Build CNN Model

The CNN model is constructed using 3 Convolutional layers and 2 Max Pooling layers before finishing with a dense layer before the softmax. This is essentially the same structure as used with MNIST but the input size and number of channels has changed.

In [ ]:
class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 32 * 32, 164)
        self.fc2 = nn.Linear(164, 10)
    
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = F.relu(self.conv3(x))
        x = self.pool(x)
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = F.softmax(self.fc2(x), dim=1)
        return x

In [ ]:
cnn_model = CNNModel().to(device)

In [ ]:
no_epochs = 200

In [ ]:
history_dict = dict()

### Train a model without Data Augmentation

In [ ]:
def train_model(model, train_loader, test_loader, loss_fn, optimizer, epochs):
    train_errors = []
    test_errors = []
    train_accuracies = []
    test_accuracies = []

    tqdm_epoch = trange(epochs)
    for epoch in tqdm_epoch:
        model.train()
        train_loss = 0.0
        correct_train = 0

        # Training
        for batch_X, batch_y in train_loader:
            # Forward pass
            outputs = model(batch_X)
            loss = loss_fn(outputs.squeeze(), batch_y)
            
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * batch_X.size(0)
            # Calculate accuracy
            predicted = torch.argmax(outputs.squeeze(), dim=1)
            targets = torch.argmax(batch_y, dim=1)
            correct_train += (predicted == targets).sum().item()

        train_loss /= len(train_loader.dataset)
        train_accuracy = 100 * correct_train / len(train_loader.dataset)
        train_errors.append(train_loss)
        train_accuracies.append(train_accuracy)
        
        # Evaluation on test set
        model.eval()
        test_loss = 0.0
        correct_test = 0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                outputs = model(batch_X)
                loss = loss_fn(outputs.squeeze(), batch_y)
                test_loss += loss.item() * batch_X.size(0)
                # Calculate accuracy
                predicted = torch.argmax(outputs.squeeze(), dim=1)
                targets = torch.argmax(batch_y, dim=1)
                correct_test += (predicted == targets).sum().item()

        test_loss /= len(test_loader.dataset)
        test_accuracy = 100 * correct_test / len(test_loader.dataset)
        test_errors.append(test_loss)
        test_accuracies.append(test_accuracy)

        tqdm_epoch.set_description(f"Epoch {epoch+1}/{epochs} \
            - Train loss: {train_loss:.4f}, Test loss: {test_loss:.4f}, \
            Train Acc: {train_accuracy:.2f}%, Test Acc: {test_accuracy:.2f}%")

    history = dict()
    history['train_loss'] = train_errors
    history['test_loss'] = test_errors
    history['train_acc'] = train_accuracies
    history['test_acc'] = test_accuracies
        
    return history

In [ ]:
# Define the optimizer (Adam in this case)
optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)

# Define the loss function (categorical cross-entropy in this case)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
history = train_model(cnn_model, train_loader, test_loader,
                      loss_fn, optimizer, no_epochs)
history_dict['Default'] = history

### Train a model with Data Aumentation

To perform data augmentation in PyTorch transforms are used

In [ ]:
class CustomAugmentations:
    def __init__(self, rotation_range, width_shift_range, height_shift_range, shear_range, zoom_range, horizontal_flip):
        self.rotation_range = rotation_range
        self.width_shift_range = width_shift_range
        self.height_shift_range = height_shift_range
        self.shear_range = shear_range
        self.zoom_range = zoom_range
        self.horizontal_flip = horizontal_flip

    def __call__(self, img):
        # Random rotation
        angle = torch.empty(1).uniform_(-self.rotation_range, self.rotation_range).item()
        img = TF.rotate(img, angle, interpolation=TF.InterpolationMode.NEAREST)

        # Random width shift
        width_shift = torch.empty(1).uniform_(-self.width_shift_range, self.width_shift_range).item()
        img = TF.affine(img, angle=0, translate=(width_shift * img.shape[2], 0), scale=1, shear=0, interpolation=TF.InterpolationMode.NEAREST)

        # Random height shift
        height_shift = torch.empty(1).uniform_(-self.height_shift_range, self.height_shift_range).item()
        img = TF.affine(img, angle=0, translate=(0, height_shift * img.shape[1]), scale=1, shear=0, interpolation=TF.InterpolationMode.NEAREST)

        # Random shear
        shear = torch.empty(1).uniform_(-self.shear_range, self.shear_range).item()
        img = TF.affine(img, angle=0, translate=(0, 0), scale=1, shear=shear, interpolation=TF.InterpolationMode.NEAREST)

        # Random zoom
        zoom = torch.empty(1).uniform_(1 - self.zoom_range, 1 + self.zoom_range).item()
        img = TF.affine(img, angle=0, translate=(0, 0), scale=zoom, shear=0, interpolation=TF.InterpolationMode.NEAREST)

        # Random horizontal flip
        if self.horizontal_flip and torch.rand(1).item() > 0.5:
            img = TF.hflip(img)
        
        return img

In [ ]:
custom_transforms = transforms.Compose([
    CustomAugmentations(rotation_range=40, 
                        width_shift_range=0.2, 
                        height_shift_range=0.2, 
                        shear_range=0.2, 
                        zoom_range=0.2, 
                        horizontal_flip=True)
])

In [ ]:
class AugmentedDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, transform=None):
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        x, y = self.dataset[idx]
        if self.transform:
            x = self.transform(x)
        return x, y

In [ ]:
aug_train_dataset = AugmentedDataset(train_dataset, 
                                     transform=custom_transforms)

In [ ]:
train_loader = DataLoader(aug_train_dataset, batch_size=128, shuffle=True)

In [ ]:
cnn_model_da = CNNModel().to(device)

In [ ]:
# Define the optimizer (Adam in this case)
optimizer = optim.Adam(cnn_model_da.parameters(), lr=0.001)

# Define the loss function (categorical cross-entropy in this case)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
history = train_model(cnn_model_da, train_loader, test_loader,
                      loss_fn, optimizer, no_epochs)
history_dict['Data Aug.'] = history

### Plot Validation Accuracy

In [ ]:
epochs = range(1, no_epochs + 1)

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))

from cycler import cycler
monochrome = (cycler('color', ['k']) * cycler('linestyle', ['-', '--', ':', '-.']))

ax.set_prop_cycle(monochrome)
for ilr in history_dict:
    valacc_hist = history_dict[ilr]
    val_acc_values = valacc_hist['test_acc']
    ax.plot(epochs, val_acc_values, label = str(ilr)+' Validation')
    
for ilr in history_dict:
    acc_hist = history_dict[ilr]
    acc_values = acc_hist['train_acc']
    ax.plot(epochs, acc_values, label = str(ilr)+' Training')
    
ax.set_xlabel('Epochs')
ax.set_ylabel('Accuracy')
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
fig.savefig('TestDataAugmentation.png', dpi=300, bbox_inches='tight')